In [ ]:
# ============================================================================
# TIỂU LUẬN: ĐÁNH GIÁ BART VÀ T5 CHO TÓM TẮT VĂN BẢN (CNN/DailyMail)
# ============================================================================
# Thiết lập: 2 epoch cho mỗi mô hình
# Lưu tokenized dataset cho cả T5 và BART
# ============================================================================

# ------------------------- 1. CÀI ĐẶT THƯ VIỆN -------------------------
!pip install transformers datasets evaluate rouge_score sacrebleu -q

# ------------------------- 2. IMPORT THƯ VIỆN -------------------------
import torch
import os
import numpy as np
from datasets import load_dataset
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration,
    BartTokenizer, BartForConditionalGeneration,
    Trainer, TrainingArguments, DataCollatorForSeq2Seq
)
import evaluate

# ------------------------- 3. LOAD DATASET -------------------------
print("Đang tải bộ dữ liệu CNN/DailyMail...")
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Lấy tập con để chạy nhanh trên Colab
train_data = dataset["train"].shuffle(seed=42).select(range(10000))
test_data = dataset["test"].shuffle(seed=42).select(range(500))

print(f"Số mẫu huấn luyện: {len(train_data)}")
print(f"Số mẫu kiểm tra: {len(test_data)}")

# ------------------------- 4. HÀM TIỀN XỬ LÝ -------------------------
def preprocess_function(examples, tokenizer, prefix=""):
    inputs = [prefix + doc for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)

    labels = tokenizer(examples["highlights"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# ------------------------- 5. HÀM HUẤN LUYỆN & ĐÁNH GIÁ -------------------------
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

def train_and_evaluate(model_name, model_class, tokenizer_class, prefix=""):
    print(f"\n{'='*60}")
    print(f"BẮT ĐẦU HUẤN LUYỆN: {model_name}")
    print(f"{'='*60}")

    # Khởi tạo tokenizer và model
    tokenizer = tokenizer_class.from_pretrained(model_name)
    model = model_class.from_pretrained(model_name)

    # Tokenize dữ liệu
    tokenized_train = train_data.map(
        lambda x: preprocess_function(x, tokenizer, prefix), batched=True
    )
    tokenized_test = test_data.map(
        lambda x: preprocess_function(x, tokenizer, prefix), batched=True
    )

    # Thiết lập tham số huấn luyện (2 EPOCH)
    training_args = TrainingArguments(
        output_dir=f"./results_{model_name.replace('/', '_')}",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=2,                         # <-- 2 EPOCH
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="no",
        fp16=True,
        report_to="none"
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        data_collator=data_collator,
    )

    # Huấn luyện
    trainer.train()

    # ------------------------- ĐÁNH GIÁ -------------------------
    print(f"\nĐang đánh giá {model_name} trên 100 mẫu test...")
    predictions, references = [], []
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    for i in range(100):
        input_text = test_data[i]["article"]
        ref_text = test_data[i]["highlights"]

        inputs = tokenizer(
            prefix + input_text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(pred_text)
        references.append([ref_text])

    # Tính ROUGE và BLEU
    rouge_score = rouge.compute(
        predictions=predictions,
        references=[r[0] for r in references]
    )
    bleu_score = bleu.compute(predictions=predictions, references=references)

    print(f"\nKẾT QUẢ {model_name}:")
    print(f"ROUGE-1 : {rouge_score['rouge1']:.4f}")
    print(f"ROUGE-2 : {rouge_score['rouge2']:.4f}")
    print(f"ROUGE-L : {rouge_score['rougeL']:.4f}")
    print(f"BLEU    : {bleu_score['bleu']:.4f}")

    return model, tokenizer, rouge_score, bleu_score

# ------------------------- 6. HUẤN LUYỆN CÁC MÔ HÌNH -------------------------
print("\n" + "#"*70)
print("# TIẾN HÀNH THỰC NGHIỆM VỚI 2 EPOCH")
print("#"*70)

# Huấn luyện T5-small
model_t5, tokenizer_t5, rouge_t5, bleu_t5 = train_and_evaluate(
    model_name="t5-small",
    model_class=T5ForConditionalGeneration,
    tokenizer_class=T5Tokenizer,
    prefix="summarize: "
)

# Huấn luyện BART-base
model_bart, tokenizer_bart, rouge_bart, bleu_bart = train_and_evaluate(
    model_name="facebook/bart-base",
    model_class=BartForConditionalGeneration,
    tokenizer_class=BartTokenizer,
    prefix=""
)

# ------------------------- 7. SO SÁNH KẾT QUẢ -------------------------
print("\n" + "="*60)
print("TỔNG KẾT SO SÁNH (2 EPOCH)")
print("="*60)
print(f"{'Mô hình':<15} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10} {'BLEU':<10}")
print("-"*55)
print(f"{'T5-small':<15} {rouge_t5['rouge1']:.4f}    {rouge_t5['rouge2']:.4f}    {rouge_t5['rougeL']:.4f}    {bleu_t5['bleu']:.4f}")
print(f"{'BART-base':<15} {rouge_bart['rouge1']:.4f}    {rouge_bart['rouge2']:.4f}    {rouge_bart['rougeL']:.4f}    {bleu_bart['bleu']:.4f}")
print("="*60)

# ------------------------- 8. LƯU DỮ LIỆU ĐÃ TOKENIZE (CẢ T5 VÀ BART) -------------------------
print("\n" + "="*60)
print("LƯU DATASET ĐÃ TOKENIZE CHO CẢ T5 VÀ BART")
print("="*60)

# Tạo thư mục lưu trữ
base_dir = "./tokenized_data"
os.makedirs(f"{base_dir}/t5/train", exist_ok=True)
os.makedirs(f"{base_dir}/t5/test", exist_ok=True)
os.makedirs(f"{base_dir}/bart/train", exist_ok=True)
os.makedirs(f"{base_dir}/bart/test", exist_ok=True)

# Tokenize và lưu cho T5
print("Đang tokenize và lưu cho T5...")
tokenized_train_t5 = train_data.map(
    lambda x: preprocess_function(x, tokenizer_t5, prefix="summarize: "), batched=True
)
tokenized_test_t5 = test_data.map(
    lambda x: preprocess_function(x, tokenizer_t5, prefix="summarize: "), batched=True
)
tokenized_train_t5.save_to_disk(f"{base_dir}/t5/train")
tokenized_test_t5.save_to_disk(f"{base_dir}/t5/test")
print(f"Đã lưu T5 tokenized data vào: {base_dir}/t5/")

# Tokenize và lưu cho BART
print("Đang tokenize và lưu cho BART...")
tokenized_train_bart = train_data.map(
    lambda x: preprocess_function(x, tokenizer_bart, prefix=""), batched=True
)
tokenized_test_bart = test_data.map(
    lambda x: preprocess_function(x, tokenizer_bart, prefix=""), batched=True
)
tokenized_train_bart.save_to_disk(f"{base_dir}/bart/train")
tokenized_test_bart.save_to_disk(f"{base_dir}/bart/test")
print(f"Đã lưu BART tokenized data vào: {base_dir}/bart/")

# ------------------------- 9. TEST SINH MẪU -------------------------
def generate_summary(model, tokenizer, text, prefix=""):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    inputs = tokenizer(prefix + text, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_length=120, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "="*60)
print("MINH HỌA TÓM TẮT (5 MẪU ĐẦU TIÊN)")
print("="*60)

for i in range(5):
    article = test_data[i]["article"]
    reference = test_data[i]["highlights"]
    print(f"\n--- MẪU {i+1} ---")
    print("Reference:", reference)
    t5_sum = generate_summary(model_t5, tokenizer_t5, article, prefix="summarize: ")
    print("T5-small :", t5_sum)
    bart_sum = generate_summary(model_bart, tokenizer_bart, article)
    print("BART-base:", bart_sum)

print("\nHOÀN THÀNH!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.6 MB/s eta 0:00:00
Đang tải bộ dữ liệu CNN/DailyMail...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

...

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Đã lưu BART tokenized data vào: ./tokenized_data/bart/

MINH HỌA TÓM TẮT (5 MẪU ĐẦU TIÊN)

--- MẪU 1 ---
Reference: CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .
He says he knows how easy it is do nothing "because I did nothing for too long"
T5-small : a medical marijuana revolution has risen 11 points in the past few years alone. Marijuana is part of the medical marijuana revolution, she says. Marijuana's seizures spur medical marijuana legislation in Georgia, she says.
BART-base: David Frum: I see signs of a revolution everywhere .
He says medical marijuana has been a big success in many areas .
Frum: Patients, doctors, doctors are now willing to stake their political futures on the issue .
Frum: Medical marijuana is a revolution in the attitudes of everyday Americans .

--- MẪU 2 ---
Reference: Child has amassed thousands of Twitter followers with 'gang life' photos .
In one video he points gun at camera as adults look on unfazed .
His tweets have prompted b

### 10.1 Gắn kết Google Drive

Bước này sẽ cho phép Colab truy cập vào các tệp trong Google Drive của bạn. Bạn sẽ cần xác thực tài khoản Google của mình.

### 10.2 Tải dữ liệu từ Google Drive

Sau khi Drive đã được gắn kết, bạn có thể tìm thấy các tệp của mình trong thư mục `/content/drive/MyDrive/`. Hãy thay thế `'path/to/your/data.csv'` bằng đường dẫn thực tế đến tệp của bạn trong Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# Ví dụ: Tải một tệp CSV từ Google Drive
# Thay thế 'path/to/your/data.csv' bằng đường dẫn thực tế của bạn
# Ví dụ: '/content/drive/MyDrive/my_data_folder/my_file.csv'
try:
    df_from_drive = pd.read_csv('/content/drive/MyDrive/path/to/your/data.csv')
    print("Đã tải dữ liệu thành công từ Google Drive:")
    display(df_from_drive.head())
except FileNotFoundError:
    print("Lỗi: Không tìm thấy tệp. Vui lòng kiểm tra lại đường dẫn tệp trong Google Drive của bạn.")
except Exception as e:
    print(f"Đã xảy ra lỗi khi tải tệp: {e}")

Lỗi: Không tìm thấy tệp. Vui lòng kiểm tra lại đường dẫn tệp trong Google Drive của bạn.
